# Qwen3-Reranker on TL Engine: Amazon SageMaker sample notebook

This notebook deploys the Qwen3-Reranker model package you subscribed to in AWS Marketplace, reranks documents on a real-time endpoint, runs a batch transform job, and deletes everything it created. It needs no data of your own, and it works for all three sizes: 0.6B, 4B and 8B.

**Before you start**

1. Subscribe to the listing for the size you want, then open the subscription's configuration page and copy the model package ARN for your Region. The notebook also finds it on its own in the Regions listed in the next cells.
2. Run the notebook with an IAM role that may create SageMaker models, endpoints and transform jobs, pass itself to SageMaker, and read and write an S3 bucket. The default SageMaker Studio execution role works.
3. Your account needs quota for one ml.g6.xlarge endpoint and one ml.g6.xlarge transform job (Service Quotas: "ml.g6.xlarge for endpoint usage" and "ml.g6.xlarge for transform job usage").

The whole notebook takes about 20 minutes. You pay for the ml.g6.xlarge instance time and the software fee for that time; the clean-up section deletes the endpoint so billing stops.

In [ ]:
# Settings. Only MODEL is required.
MODEL = "4b"               # the listing you subscribed to: "0.6b", "4b" or "8b"
MODEL_PACKAGE_ARN = None   # optional: the ARN from your subscription's configuration page
ROLE = None                # optional: an IAM role ARN for SageMaker; default: the role this notebook runs as
BUCKET = None              # optional: S3 bucket for the batch job; default: sagemaker-<region>-<account>
INSTANCE_TYPE = "ml.g6.xlarge"
RUN_BATCH = True           # set to False to skip the batch transform job
SAMPLES_DIR = None         # optional: a folder to save the requests and responses in as JSON

In [ ]:
import json
import os
import time

import boto3

session = boto3.Session()
region = session.region_name
account = session.client("sts").get_caller_identity()["Account"]
sm = session.client("sagemaker")
runtime = session.client("sagemaker-runtime")
s3 = session.client("s3")

# The model package ARN of each listing per Region. A Region missing here works too: paste the ARN from
# your subscription's configuration page into MODEL_PACKAGE_ARN above.
MODEL_PACKAGES = {
}

if MODEL_PACKAGE_ARN is None:
    try:
        MODEL_PACKAGE_ARN = MODEL_PACKAGES[region][MODEL]
    except KeyError:
        raise ValueError(
            f"No model package ARN for {MODEL} in {region} in this notebook. Copy it from your subscription's "
            "configuration page in AWS Marketplace and set MODEL_PACKAGE_ARN in the first code cell."
        )

if ROLE is None:
    # In SageMaker Studio and on notebook instances the notebook runs as an assumed role:
    # arn:aws:sts::<account>:assumed-role/<role name>/<session>
    caller = session.client("sts").get_caller_identity()["Arn"]
    if ":assumed-role/" not in caller:
        raise ValueError("Set ROLE to the ARN of an IAM role that SageMaker can assume.")
    role_name = caller.split(":assumed-role/")[1].split("/")[0]
    try:
        ROLE = session.client("iam").get_role(RoleName=role_name)["Role"]["Arn"]
    except Exception:
        ROLE = f"arn:aws:iam::{account}:role/{role_name}"

if BUCKET is None:
    BUCKET = f"sagemaker-{region}-{account}"

name = f"qwen3-reranker-{MODEL.replace('.', '')}-" + time.strftime("%Y%m%d-%H%M%S")
print("Region:       ", region)
print("Model package:", MODEL_PACKAGE_ARN)
print("Role:         ", ROLE)
print("Resource name:", name)

## 1. The input

A request holds a query and the documents to rank against it. Send plain text: the container applies the model's prompt template and tokenizes, so no preprocessing is needed.

| Field | Type | Meaning |
|---|---|---|
| `query` | string | The search query or question |
| `documents` | list of strings | The candidate passages, in any order |

A query and one document may hold up to 33,000 tokens together. Real-time and batch take the same format.

In [ ]:
requests = {
    "capital": {
        "query": "What is the capital of France?",
        "documents": [
            "France is a country in Europe.",
            "Paris is the capital of France.",
            "Berlin is the capital of Germany.",
        ],
    },
    "password": {
        "query": "How do I reset my password?",
        "documents": [
            "Invoices are sent on the first working day of each month.",
            "To choose a new password, open Settings, then Security, and select Reset password. "
            "A link is sent to your email address.",
            "Our offices are closed on public holidays.",
        ],
    },
}
# The document that answers each query, used to check the results below.
expected_best = {"capital": 1, "password": 1}

## 2. Create the model and a real-time endpoint

The endpoint takes about 5 to 7 minutes to come into service: SageMaker starts a GPU instance, pulls the container and loads the weights. The model runs with network isolation, so your queries and documents never leave your account.

In [ ]:
sm.create_model(
    ModelName=name,
    ExecutionRoleArn=ROLE,
    PrimaryContainer={"ModelPackageName": MODEL_PACKAGE_ARN},
    EnableNetworkIsolation=True,
)
sm.create_endpoint_config(
    EndpointConfigName=name,
    ProductionVariants=[{
        "VariantName": "AllTraffic",
        "ModelName": name,
        "InstanceType": INSTANCE_TYPE,
        "InitialInstanceCount": 1,
    }],
)
sm.create_endpoint(EndpointName=name, EndpointConfigName=name)
started = time.time()
sm.get_waiter("endpoint_in_service").wait(EndpointName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 60})
print(f"Endpoint {name} in service after {(time.time() - started) / 60:.1f} minutes")

## 3. Rerank in real time

The response lists the documents from most to least relevant. `index` is the document's position in your request and `relevance_score` is between 0 and 1.

In [ ]:
def rerank(request):
    response = runtime.invoke_endpoint(
        EndpointName=name,
        ContentType="application/json",
        Accept="application/json",
        Body=json.dumps(request),
    )
    return json.loads(response["Body"].read())


responses = {}
for label, request in requests.items():
    result = rerank(request)
    responses[label] = result
    print(f"{request['query']}  ({result['usage']['total_tokens']} tokens)")
    for r in result["results"]:
        print(f"  {r['relevance_score']:.4f}  {request['documents'][r['index']]}")
    assert result["results"][0]["index"] == expected_best[label], f"Unexpected top document for {label}"
print("\nThe right document came first for every query.")

if SAMPLES_DIR:
    os.makedirs(SAMPLES_DIR, exist_ok=True)
    with open(os.path.join(SAMPLES_DIR, "input.json"), "w") as f:
        json.dump(requests["capital"], f, indent=2)
    with open(os.path.join(SAMPLES_DIR, f"output-{MODEL}.json"), "w") as f:
        json.dump(responses["capital"], f, indent=2)

## 4. Batch transform

Each input file is one request in the same JSON format. For each file the job writes `<file name>.out` with the same response as the endpoint. The job starts its own instance, so it takes about 10 minutes.

In [ ]:
if RUN_BATCH:
    try:
        s3.head_bucket(Bucket=BUCKET)
    except Exception:
        config = {} if region == "us-east-1" else {"CreateBucketConfiguration": {"LocationConstraint": region}}
        s3.create_bucket(Bucket=BUCKET, **config)
    prefix = f"{name}/"
    for label, request in requests.items():
        s3.put_object(Bucket=BUCKET, Key=f"{prefix}input/{label}.json", Body=json.dumps(request))

    sm.create_transform_job(
        TransformJobName=name,
        ModelName=name,
        TransformInput={
            "DataSource": {"S3DataSource": {"S3DataType": "S3Prefix", "S3Uri": f"s3://{BUCKET}/{prefix}input/"}},
            "ContentType": "application/json",
        },
        TransformOutput={"S3OutputPath": f"s3://{BUCKET}/{prefix}output/", "Accept": "application/json"},
        TransformResources={"InstanceType": INSTANCE_TYPE, "InstanceCount": 1},
    )
    sm.get_waiter("transform_job_completed_or_stopped").wait(
        TransformJobName=name, WaiterConfig={"Delay": 30, "MaxAttempts": 80}
    )
    job = sm.describe_transform_job(TransformJobName=name)
    assert job["TransformJobStatus"] == "Completed", job.get("FailureReason")

    for label, request in requests.items():
        body = s3.get_object(Bucket=BUCKET, Key=f"{prefix}output/{label}.json.out")["Body"].read()
        batch = json.loads(body)
        best = batch["results"][0]
        print(f"{label}: {request['documents'][best['index']]} ({best['relevance_score']:.4f})")
        # Batch and real time run the same container, so they rank the same way.
        assert [r["index"] for r in batch["results"]] == [r["index"] for r in responses[label]["results"]]
    print("\nBatch transform ranked every request as the endpoint did.")

## 5. Clean up

Delete the endpoint, its configuration and the model so that billing stops, and remove the batch job's files.

In [ ]:
sm.delete_endpoint(EndpointName=name)
sm.delete_endpoint_config(EndpointConfigName=name)
sm.delete_model(ModelName=name)
if RUN_BATCH:
    for page in s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix=f"{name}/"):
        for obj in page.get("Contents", []):
            s3.delete_object(Bucket=BUCKET, Key=obj["Key"])
print("Deleted", name)

To end the subscription itself, open **AWS Marketplace → Manage subscriptions** in the AWS console, choose the listing and cancel it.